# 06.9 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/06-unsupervised/09-market-basket/solutions.ipynb)

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

baskets = [
    ["tea", "cake", "mug"],
    ["tea", "cake"],
    ["tea", "mug"],
    ["cake", "candle"],
    ["tea", "cake", "card"],
    ["candle", "card"],
    ["tea", "cake", "candle"],
    ["mug", "card"],
]
items = sorted({i for b in baskets for i in b})
onehot = pd.DataFrame([[i in b for i in items] for b in baskets], columns=items)
onehot.index = [f"B{k}" for k in range(1, 9)]
onehot.astype(int)

,cake,candle,card,mug,tea
B1,1,0,0,1,1
B2,1,0,0,0,1
B3,0,0,0,1,1
B4,1,1,0,0,0
B5,1,0,1,0,1
B6,0,1,1,0,0
B7,1,1,0,0,1
B8,0,0,1,1,0


## Exercise 1

- support(cake, tea) = 4/8 = 0.5 (the same set as tea, cake)
- confidence(cake → tea) = 0.5 / support(cake) = 0.5 / 0.625 = **0.8**
- lift(cake → tea) = 0.8 / support(tea) = 0.8 / 0.625 = **1.28**, identical to tea → cake.

Card is in B5, B6, B8; mug is in B1, B3, B8. Both only in B8: support = 1/8 = 0.125.

- confidence(card → mug) = 0.125 / 0.375 = **0.333**
- lift(card → mug) = 0.333 / 0.375 = **0.889**

**Lift is symmetric**: lift(A → B) = support(A, B) / (support(A) × support(B)), which does not care about order. **Confidence is not** in general: it divides by the antecedent's support. Here confidence happens to match both ways because tea and cake (and card and mug) have equal supports.

In [2]:
s = lambda *t: onehot[list(t)].all(axis=1).mean()
for a, b in [("cake", "tea"), ("tea", "cake"), ("card", "mug"), ("mug", "card"), ("tea", "mug"), ("mug", "tea")]:
    conf = s(a, b) / s(a)
    print(f"{a:>5} -> {b:<5} confidence {conf:.3f}  lift {conf / s(b):.3f}")

 cake -> tea   confidence 0.800  lift 1.280
  tea -> cake  confidence 0.800  lift 1.280
 card -> mug   confidence 0.333  lift 0.889
  mug -> card  confidence 0.333  lift 0.889
  tea -> mug   confidence 0.400  lift 1.067
  mug -> tea   confidence 0.667  lift 1.067


The last two lines show confidence differing by direction (tea → mug 0.4, mug → tea 0.667) while lift stays the same (1.067).

## Exercise 2

In [3]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"], **{"Customer ID": sales["Customer ID"].astype(int)})
print(f"{len(sales):,} clean transaction lines, {sales['Customer ID'].nunique():,} customers")

from mlxtend.frequent_patterns import apriori, association_rules

def basket_matrix(df, drop_postage=True):
    """One row per invoice, one True/False column per product."""
    b = df.groupby(["Invoice", "Description"])["Quantity"].sum().unstack(fill_value=0) > 0
    if drop_postage:
        b = b.drop(columns=[c for c in b.columns if "POSTAGE" in str(c)])
    return b

def show(rules, n=8):
    out = rules.sort_values("lift", ascending=False).head(n).copy()
    out["if bought"] = out["antecedents"].apply(lambda s: " + ".join(sorted(s)))
    out["then also"] = out["consequents"].apply(lambda s: " + ".join(sorted(s)))
    return out[["if bought", "then also", "support", "confidence", "lift"]].round(3)

germany = basket_matrix(sales[sales["Country"] == "Germany"])
rules_de = association_rules(apriori(germany, min_support=0.05, use_colnames=True), metric="lift", min_threshold=1.0)
print(f"{germany.shape[0]} invoices, {len(rules_de)} rules")
show(rules_de, n=8)

392,693 clean transaction lines, 4,338 customers


457 invoices, 18 rules


,if bought,then also,support,confidence,lift
10,WOODLAND CHARLOTTE BAG,RED RETROSPOT CHARLOTTE BAG,0.059,0.466,6.648
11,RED RETROSPOT CHARLOTTE BAG,WOODLAND CHARLOTTE BAG,0.059,0.844,6.648
0,PLASTERS IN TIN CIRCUS PARADE,PLASTERS IN TIN WOODLAND ANIMALS,0.068,0.585,4.243
1,PLASTERS IN TIN WOODLAND ANIMALS,PLASTERS IN TIN CIRCUS PARADE,0.068,0.492,4.243
6,PLASTERS IN TIN SPACEBOY,PLASTERS IN TIN WOODLAND ANIMALS,0.061,0.571,4.145
7,PLASTERS IN TIN WOODLAND ANIMALS,PLASTERS IN TIN SPACEBOY,0.061,0.444,4.145
12,ROUND SNACK BOXES SET OF 4 FRUITS,ROUND SNACK BOXES SET OF4 WOODLAND,0.131,0.833,3.400
13,ROUND SNACK BOXES SET OF4 WOODLAND,ROUND SNACK BOXES SET OF 4 FRUITS,0.131,0.536,3.400


Same kind of rules: product **families in different designs**. Charlotte bags in red and woodland prints, plasters in tins with different motifs, snack boxes in two patterns. Lifts are lower than in France (top about 6.6 vs 14), partly because German customers buy a wider range, so no single pair dominates.

## Exercise 3

In [4]:
fr = sales[sales["Country"] == "France"]
halves = {"H1": fr[fr["InvoiceDate"] < "2011-07-01"], "H2": fr[fr["InvoiceDate"] >= "2011-07-01"]}
found = {}
for name, part in halves.items():
    b = basket_matrix(part)
    r = association_rules(apriori(b, min_support=0.05, use_colnames=True), metric="lift", min_threshold=1.0)
    found[name] = set(zip(r["antecedents"], r["consequents"]))
    print(f"{name}: {b.shape[0]} invoices, {len(r)} rules")
shared = found["H1"] & found["H2"]
print(f"{len(shared)} of {len(found['H1'])} first-half rules also appear in the second half")

H1: 179 invoices, 192 rules
H2: 210 invoices, 200 rules
50 of 192 first-half rules also appear in the second half


Only about a quarter of the first half's rules (50 of 192) reappear in the second half, even though each half has about 200 rules. Rules about stable sets (matching tableware, colour variants of the same product) tend to survive; rules driven by a seasonal range or a single large customer do not. Before acting on a rule, check it on fresh data, exactly as you would test a model on a hold-out set.